# AlphaGenome Pipeline — Maternal Health Multi-Omics Study
## RMIT University | Digital Health and Bioinformatics Lab
### Supervised by: Associate Prof. Sonika Tyagi & Murali

## 1. Imports & Environment Setup

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

from alphagenome.data import genome
from alphagenome.models import dna_client
from alphagenome.models.dna_output import OutputType

# Load API key from .env file
load_dotenv()
API_KEY = os.getenv("ALPHAGENOME_API_KEY")

if not API_KEY or API_KEY == "your_key_here":
    print("⚠️  API key not set. Add your key to the .env file before running predictions.")
else:
    print("✅ API key loaded.")

✅ API key loaded.


## 2. Gene Input

Two input modes supported:
- **`'genes'`** — provide a list of gene names, coordinates looked up automatically from Ensembl
- **`'csv'`** — provide a CSV file with columns: `chromosome`, `start`, `end`, `gene_name`

In [2]:
import requests
import time

# ── INPUT MODE ───────────────────────────────────────────────────────────────
INPUT_MODE = 'genes'   # 'genes' or 'csv'

# Option A — gene names (used when INPUT_MODE = 'genes')
GENE_NAMES = [
    'ESR1', 'JUND', 'SOCS3', 'GAS5',
    'PVT1', 'SNHG16', 'RELA', 'NFKB1',
    'ERBB2',
]

# Option B — CSV file path (used when INPUT_MODE = 'csv')
# Expected columns: chromosome, start, end, gene_name
CSV_PATH = 'data/genes.csv'
# ─────────────────────────────────────────────────────────────────────────────

def lookup_gene(symbol):
    """Fetch GRCh38 coordinates for a gene symbol from the Ensembl REST API."""
    url = f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{symbol}"
    r = requests.get(url, headers={"Content-Type": "application/json"}, timeout=15)
    if r.status_code != 200:
        print(f"  ❌ {symbol}: not found (HTTP {r.status_code})")
        return None
    d = r.json()
    return {
        "gene":       d.get("display_name", symbol),
        "chromosome": f"chr{d['seq_region_name']}",
        "start":      d["start"],
        "end":        d["end"],
    }

def load_from_csv(path):
    """Load gene coordinates from a 4-column CSV file and validate format."""
    raw = pd.read_csv(path)
    required = {'chromosome', 'start', 'end', 'gene_name'}
    missing = required - set(raw.columns)
    if missing:
        raise ValueError(f"CSV missing required columns: {missing}. Expected: chromosome, start, end, gene_name")
    if not raw['chromosome'].astype(str).str.startswith('chr').all():
        raise ValueError("'chromosome' column must start with 'chr' (e.g. chr6, chr19)")
    if raw[['chromosome', 'start', 'end', 'gene_name']].isnull().any().any():
        raise ValueError("CSV contains missing values — check all rows are complete")
    df_out = pd.DataFrame({
        "gene":       raw["gene_name"],
        "chromosome": raw["chromosome"].astype(str),
        "start":      raw["start"].astype(int),
        "end":        raw["end"].astype(int),
    })
    return df_out

# Run the selected input mode
if INPUT_MODE == 'genes':
    print(f"Mode: gene name lookup via Ensembl (GRCh38)")
    genes_data = []
    for name in GENE_NAMES:
        coords = lookup_gene(name)
        if coords:
            genes_data.append(coords)
            print(f"  ✅ {coords['gene']}: {coords['chromosome']}:{coords['start']:,}–{coords['end']:,}")
        time.sleep(0.3)
    df = pd.DataFrame(genes_data)

elif INPUT_MODE == 'csv':
    print(f"Mode: loading coordinates from CSV — {CSV_PATH}")
    df = load_from_csv(CSV_PATH)
    for _, row in df.iterrows():
        print(f"  ✅ {row['gene']}: {row['chromosome']}:{row['start']:,}–{row['end']:,}")

else:
    raise ValueError(f"Unknown INPUT_MODE '{INPUT_MODE}'. Use 'genes' or 'csv'.")

df["length_bp"] = df["end"] - df["start"]
print(f"\nLoaded {len(df)} genes")
df

Mode: gene name lookup via Ensembl (GRCh38)
  ✅ ESR1: chr6:151,651,284–152,129,619
  ✅ JUND: chr19:18,279,694–18,281,622
  ✅ SOCS3: chr17:78,356,778–78,360,930
  ✅ GAS5: chr1:173,851,424–173,868,940
  ✅ PVT1: chr8:127,794,513–128,188,202
  ✅ SNHG16: chr17:76,557,191–76,714,384
  ✅ RELA: chr11:65,653,599–65,663,090
  ✅ NFKB1: chr4:102,500,937–102,617,302
  ✅ ERBB2: chr17:39,687,914–39,730,426

Loaded 9 genes


,gene,chromosome,start,end,length_bp
0,ESR1,chr6,151651284,152129619,478335
1,JUND,chr19,18279694,18281622,1928
2,SOCS3,chr17,78356778,78360930,4152
3,GAS5,chr1,173851424,173868940,17516
4,PVT1,chr8,127794513,128188202,393689
5,SNHG16,chr17,76557191,76714384,157193
6,RELA,chr11,65653599,65663090,9491
7,NFKB1,chr4,102500937,102617302,116365
8,ERBB2,chr17,39687914,39730426,42512


## 3. AlphaGenome API — Predictions

In [4]:
# Create the AlphaGenome model client
model = dna_client.create(api_key=API_KEY)

# Modalities we want for every gene
REQUESTED_OUTPUTS = [
    OutputType.RNA_SEQ,
    OutputType.ATAC,
    OutputType.CHIP_HISTONE,
    OutputType.CHIP_TF,
    OutputType.SPLICE_JUNCTIONS,
    OutputType.CONTACT_MAPS,
]

# Supervisor confirmed: uterus tissue focus
UTERUS_ONTOLOGIES = ['UBERON:0000995', 'CL:0002601']

# Set to None initially to get all tracks — we filter downstream by ontology term
ONTOLOGY_TERMS = None

# Extend gene boundaries by 10% upstream and downstream to capture promoter regions
BOUNDARY_EXTENSION = 0.10

# AlphaGenome only accepts these exact sequence lengths (in bp)
SUPPORTED_LENGTHS = [16384, 131072, 524288, 1048576]

def snap_to_supported_length(chromosome, start, end):
    """Extend gene boundaries by 10% each side, then snap to nearest supported length."""
    gene_length = end - start
    extension = int(gene_length * BOUNDARY_EXTENSION)
    ext_start = max(0, start - extension)
    ext_end = end + extension
    ext_length = ext_end - ext_start
    seq_length = next((l for l in SUPPORTED_LENGTHS if l >= ext_length), SUPPORTED_LENGTHS[-1])
    center = (ext_start + ext_end) // 2
    new_start = max(0, center - seq_length // 2)
    new_end = new_start + seq_length
    return new_start, new_end

def predict_gene(row):
    """Run AlphaGenome predictions for a single gene locus."""
    start, end = snap_to_supported_length(row["chromosome"], row["start"], row["end"])
    interval = genome.Interval(
        chromosome=row["chromosome"],
        start=start,
        end=end,
    )
    output = model.predict_interval(
        interval=interval,
        requested_outputs=REQUESTED_OUTPUTS,
        ontology_terms=ONTOLOGY_TERMS,
    )
    return output

# Run predictions for all genes and store results
results = {}
for _, row in df.iterrows():
    print(f"Predicting {row['gene']} ({row['chromosome']}:{row['start']}-{row['end']})...")
    results[row["gene"]] = predict_gene(row)
    print(f"  ✅ Done")

print(f"\nAll {len(results)} genes predicted successfully.")

Predicting ESR1 (chr6:151651284-152129619)...
  ✅ Done
Predicting JUND (chr19:18279694-18281622)...
  ✅ Done
Predicting SOCS3 (chr17:78356778-78360930)...
  ✅ Done
Predicting GAS5 (chr1:173851424-173868940)...
  ✅ Done
Predicting PVT1 (chr8:127794513-128188202)...
  ✅ Done
Predicting SNHG16 (chr17:76557191-76714384)...
  ✅ Done
Predicting RELA (chr11:65653599-65663090)...
  ✅ Done
Predicting NFKB1 (chr4:102500937-102617302)...
  ✅ Done
Predicting ERBB2 (chr17:39687914-39730426)...
  ✅ Done

All 9 genes predicted successfully.


## 4. Outputs & Figures

In [5]:
# Sanity check plot — gene locus lengths
plt.figure(figsize=(10, 5))
sns.barplot(data=df, x="gene", y="length_bp", palette="viridis")
plt.title("Genomic Locus Length per Gene")
plt.xlabel("Gene")
plt.ylabel("Length (bp)")
plt.tight_layout()
plt.savefig("outputs/figures/gene_locus_lengths.png", dpi=150)
plt.close()

# TODO: Add per-modality plots once predictions are confirmed working
# e.g. RNA-seq track plots, ChIP-seq signal, contact maps heatmaps, splicing arcs

/var/folders/lk/5n8c7qpx0hzd5nzp5ns6mz_40000gn/T/ipykernel_11106/1556953428.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x="gene", y="length_bp", palette="viridis")


In [6]:
# Extract and save metadata from the first gene — metadata is identical for all genes
# (same model, same ontology_terms=None)
first_gene = df['gene'].iloc[0]
output = results[first_gene]

output.chip_histone.metadata.to_csv('outputs/predictions/chipSeq_metadata.csv', index=True)
print("chip_histone tracks:", len(output.chip_histone.metadata))

output.rna_seq.metadata.to_csv('outputs/predictions/rnaSeq_metadata.csv', index=True)
print("rna_seq tracks:     ", len(output.rna_seq.metadata))

output.chip_tf.metadata.to_csv('outputs/predictions/chipTF_metadata.csv', index=True)
print("chip_tf tracks:     ", len(output.chip_tf.metadata))

output.atac.metadata.to_csv('outputs/predictions/atac_metadata.csv', index=True)
print("atac tracks:        ", len(output.atac.metadata))

output.splice_junctions.metadata.to_csv('outputs/predictions/spliceJunction_metadata.csv', index=True)
print("splice_junctions tracks:", len(output.splice_junctions.metadata))

output.contact_maps.metadata.to_csv('outputs/predictions/contactMaps_metadata.csv', index=True)
print("contact_maps tracks:    ", len(output.contact_maps.metadata))

chip_histone tracks: 1116
rna_seq tracks:      667
chip_tf tracks:      1617
atac tracks:         167
splice_junctions tracks: 367
contact_maps tracks:     28


## 5. Save Prediction Values

In [7]:
# Save mean and variance of prediction signal per gene per track for each modality
# Rows = genes, Columns = tracks (identified by track name from metadata)

def save_mean_predictions(results, modality_attr, metadata_col, filename):
    """Build a gene x track matrix of mean predicted signal and save to CSV."""
    rows = {}
    for gene, output in results.items():
        modality = getattr(output, modality_attr)
        if modality is None:
            continue
        mean_signal = modality.values.mean(axis=0)
        track_names = modality.metadata[metadata_col].tolist()
        rows[gene] = dict(zip(track_names, mean_signal))
    df_out = pd.DataFrame(rows).T
    df_out.index.name = "gene"
    df_out.to_csv(f"outputs/predictions/{filename}")
    print(f"Saved {filename} — shape: {df_out.shape}")
    return df_out

def save_variance_predictions(results, modality_attr, metadata_col, filename):
    """Build a gene x track matrix of signal variance across positions and save to CSV."""
    rows = {}
    for gene, output in results.items():
        modality = getattr(output, modality_attr)
        if modality is None:
            continue
        var_signal = modality.values.var(axis=0)
        track_names = modality.metadata[metadata_col].tolist()
        rows[gene] = dict(zip(track_names, var_signal))
    df_out = pd.DataFrame(rows).T
    df_out.index.name = "gene"
    df_out.to_csv(f"outputs/predictions/{filename}")
    print(f"Saved {filename} — shape: {df_out.shape}")
    return df_out

# Save mean signal
rna_df       = save_mean_predictions(results, "rna_seq",      "name", "rnaSeq_predictions.csv")
chip_hist_df = save_mean_predictions(results, "chip_histone", "name", "chipHistone_predictions.csv")
chip_tf_df   = save_mean_predictions(results, "chip_tf",      "name", "chipTF_predictions.csv")
atac_df      = save_mean_predictions(results, "atac",         "name", "atac_predictions.csv")

print()

# Save variance (signal concentration across positions — high = signal is peaked, not flat)
rna_var_df       = save_variance_predictions(results, "rna_seq",      "name", "rnaSeq_variance.csv")
chip_hist_var_df = save_variance_predictions(results, "chip_histone", "name", "chipHistone_variance.csv")
chip_tf_var_df   = save_variance_predictions(results, "chip_tf",      "name", "chipTF_variance.csv")
atac_var_df      = save_variance_predictions(results, "atac",         "name", "atac_variance.csv")

Saved rnaSeq_predictions.csv — shape: (9, 371)
Saved chipHistone_predictions.csv — shape: (9, 1116)
Saved chipTF_predictions.csv — shape: (9, 1617)
Saved atac_predictions.csv — shape: (9, 167)

Saved rnaSeq_variance.csv — shape: (9, 371)
Saved chipHistone_variance.csv — shape: (9, 1116)
Saved chipTF_variance.csv — shape: (9, 1617)
Saved atac_variance.csv — shape: (9, 167)


## 6. Uterus-Focused Analysis

Supervisor confirmed: focus on **uterus tissue**.

Filter all saved predictions down to uterus-relevant tracks using two ontology terms:
- `UBERON:0000995` — uterus (tissue)
- `CL:0002601` — uterine smooth muscle cell (cell type)

### Figure 0 — All Modalities Stacked (Genomic Track View)

This replicates the figure style from the AlphaGenome paper — all predicted modalities shown as signal tracks along the gene locus (x-axis = genomic position).

This is different from the heatmaps below: instead of one summary value per gene, each track shows the **predicted signal at every base pair** across the locus.

In [8]:
from alphagenome.visualization import plot_components

# Helper: filter a TrackData to uterus tracks, fall back to all if none found
def uterus_tracks(tdata):
    mask = tdata.metadata['ontology_curie'].isin(UTERUS_ONTOLOGIES).values
    return tdata.filter_tracks(mask) if mask.sum() > 0 else tdata

# Generate the stacked all-modalities figure for every gene in the list
for gene in df['gene']:
    output = results[gene]
    row = df[df['gene'] == gene].iloc[0]
    gene_interval = genome.Interval(row['chromosome'], row['start'], row['end'])

    rna_u  = uterus_tracks(output.rna_seq)
    atac_u = uterus_tracks(output.atac)
    hist_u = uterus_tracks(output.chip_histone)
    tf_u   = uterus_tracks(output.chip_tf)

    sj_mask = output.splice_junctions.metadata['ontology_curie'].isin(UTERUS_ONTOLOGIES).values
    sj_u = output.splice_junctions.filter_tracks(sj_mask) if sj_mask.sum() > 0 else output.splice_junctions

    # Contact maps: use HeLa-S3 as single representative track
    cm_meta = output.contact_maps.metadata.reset_index(drop=True)
    hela_mask = [r['biosample_name'] == 'HeLa-S3' for _, r in cm_meta.iterrows()]
    cm_hela = output.contact_maps.filter_tracks(hela_mask)

    components = [
        plot_components.Tracks(tdata=rna_u,  ylabel_template='RNA-seq\n{biosample_name}'),
        plot_components.Tracks(tdata=atac_u, ylabel_template='ATAC\n{biosample_name}'),
        plot_components.Tracks(tdata=hist_u, ylabel_template='Histone: {name}'),
        plot_components.Tracks(tdata=tf_u,   ylabel_template='TF: {name}'),
        plot_components.Sashimi(sj_u, ylabel_template='Splice junctions\n{biosample_name}',
                                filter_threshold=0.01, normalize_values=False),
        plot_components.ContactMaps(tdata=cm_hela, ylabel_template='{biosample_name}', cmap='autumn_r', vmax=1.0),
    ]

    plot = plot_components.plot(
        components,
        interval=gene_interval,
        title=f'AlphaGenome Predictions — {gene} (Uterus tissue / HeLa-S3 contact map)',
    )
    plt.savefig(f'outputs/figures/allModalities_{gene}.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: outputs/figures/allModalities_{gene}.png")

Saved: outputs/figures/allModalities_ESR1.png
Saved: outputs/figures/allModalities_JUND.png
Saved: outputs/figures/allModalities_SOCS3.png
Saved: outputs/figures/allModalities_GAS5.png
Saved: outputs/figures/allModalities_PVT1.png
Saved: outputs/figures/allModalities_SNHG16.png
Saved: outputs/figures/allModalities_RELA.png
Saved: outputs/figures/allModalities_NFKB1.png
Saved: outputs/figures/allModalities_ERBB2.png


In [9]:
# Load saved prediction matrices and metadata
rna_pred  = pd.read_csv('outputs/predictions/rnaSeq_predictions.csv', index_col=0)
hist_pred = pd.read_csv('outputs/predictions/chipHistone_predictions.csv', index_col=0)
tf_pred   = pd.read_csv('outputs/predictions/chipTF_predictions.csv', index_col=0)
atac_pred = pd.read_csv('outputs/predictions/atac_predictions.csv', index_col=0)

rna_meta  = pd.read_csv('outputs/predictions/rnaSeq_metadata.csv', index_col=0)
hist_meta = pd.read_csv('outputs/predictions/chipSeq_metadata.csv', index_col=0)
tf_meta   = pd.read_csv('outputs/predictions/chipTF_metadata.csv', index_col=0)
atac_meta = pd.read_csv('outputs/predictions/atac_metadata.csv', index_col=0)

# Ontology terms confirmed by supervisor
UTERUS_ONTOLOGIES = ['UBERON:0000995', 'CL:0002601']

def filter_to_uterus(pred_df, meta_df):
    """Return columns of pred_df that correspond to uterus tracks in metadata."""
    uterus_names = (
        meta_df[meta_df['ontology_curie'].isin(UTERUS_ONTOLOGIES)]['name']
        .drop_duplicates()
        .tolist()
    )
    # Keep only columns that exist in the predictions file
    cols = [c for c in uterus_names if c in pred_df.columns]
    return pred_df[cols]

rna_uterus  = filter_to_uterus(rna_pred,  rna_meta)
hist_uterus = filter_to_uterus(hist_pred, hist_meta)
tf_uterus   = filter_to_uterus(tf_pred,   tf_meta)
atac_uterus = filter_to_uterus(atac_pred, atac_meta)

print(f"RNA-seq uterus tracks:     {rna_uterus.shape[1]}")
print(f"ChIP-histone uterus tracks: {hist_uterus.shape[1]}")
print(f"ChIP-TF uterus tracks:      {tf_uterus.shape[1]}")
print(f"ATAC uterus tracks:         {atac_uterus.shape[1]}")
print()
print("ATAC tracks:")
for c in atac_uterus.columns: print(f"  {c}")

RNA-seq uterus tracks:     3
ChIP-histone uterus tracks: 2
ChIP-TF uterus tracks:      2
ATAC uterus tracks:         1

ATAC tracks:
  UBERON:0000995 ATAC-seq


### Figure 5 — (Variance) Predicted Signal Variance in Uterus Tissue (RNA-seq)

Variance measures how **concentrated** or **peaked** the signal is across the gene locus.

- **High variance** = signal is focused at specific positions (e.g. a sharp peak at the promoter or exon) — biologically meaningful, gene has distinct active regions
- **Low variance** = signal is flat/spread across the whole locus — less specific regulatory activity

This helps correct for locus size bias: a large gene like ESR1 may have a low *mean* signal simply because signal is diluted across 473kb, but its *variance* reveals whether there are strong peaks within that locus.

In [10]:
# Load variance CSVs
rna_var   = pd.read_csv('outputs/predictions/rnaSeq_variance.csv', index_col=0)
hist_var  = pd.read_csv('outputs/predictions/chipHistone_variance.csv', index_col=0)
atac_var  = pd.read_csv('outputs/predictions/atac_variance.csv', index_col=0)

# Filter to uterus tracks
rna_var_uterus  = filter_to_uterus(rna_var,  rna_meta)
hist_var_uterus = filter_to_uterus(hist_var, hist_meta)
atac_var_uterus = filter_to_uterus(atac_var, atac_meta)

RNA_LABELS = {
    'CL:0002601 total RNA-seq':                      'Uterine smooth\nmuscle (total)',
    'UBERON:0000995 total RNA-seq':                  'Uterus\n(total)',
    'UBERON:0000995 gtex Uterus polyA plus RNA-seq': 'Uterus GTEx\n(polyA+)',
}

HIST_LABELS = {
    'UBERON:0000995 Histone ChIP-seq H3K27ac': 'H3K27ac\n(active enhancer)',
    'UBERON:0000995 Histone ChIP-seq H3K4me3': 'H3K4me3\n(active promoter)',
}

# RNA-seq variance heatmap
rna_var_plot = np.log1p(rna_var_uterus).rename(columns=RNA_LABELS)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(rna_var_plot, annot=True, fmt='.2f', cmap='PuBu', linewidths=0.5,
            cbar_kws={'label': 'log1p(signal variance across positions)'}, ax=ax)
ax.set_title('Predicted RNA-seq Signal Variance — Uterus Tissue\n(Higher = more concentrated/peaked signal)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Track', fontsize=12)
ax.set_ylabel('Gene', fontsize=12)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig('outputs/figures/rnaSeq_variance_uterus_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: outputs/figures/rnaSeq_variance_uterus_heatmap.png")

# Histone variance bar chart
hist_var_plot = hist_var_uterus.rename(columns=HIST_LABELS).reset_index()
hist_var_long = hist_var_plot.melt(id_vars='gene', var_name='Histone Mark', value_name='Variance')

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=hist_var_long, x='gene', y='Variance', hue='Histone Mark',
            palette=['#e74c3c', '#3498db'], ax=ax)
ax.set_title('Predicted Histone Signal Variance — Uterus Tissue\n(Higher = more localised peak)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Gene', fontsize=12)
ax.set_ylabel('Signal Variance Across Positions', fontsize=12)
ax.legend(title='Histone Mark', fontsize=10)
sns.despine()
plt.tight_layout()
plt.savefig('outputs/figures/chipHistone_variance_uterus.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: outputs/figures/chipHistone_variance_uterus.png")


Saved: outputs/figures/rnaSeq_variance_uterus_heatmap.png
Saved: outputs/figures/chipHistone_variance_uterus.png


### Figure 1 — Predicted RNA-seq Signal in Uterus Tissue

Each cell shows the mean predicted RNA-seq signal for a gene (row) in a specific uterus track (column).
Values are log1p-transformed to handle the skewed distribution typical in genomics data.

In [11]:
# Shorten track names for display
RNA_LABELS = {
    'CL:0002601 total RNA-seq':                          'Uterine smooth\nmuscle (total)',
    'UBERON:0000995 total RNA-seq':                      'Uterus\n(total)',
    'UBERON:0000995 gtex Uterus polyA plus RNA-seq':     'Uterus GTEx\n(polyA+)',
}

rna_plot = np.log1p(rna_uterus).rename(columns=RNA_LABELS)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    rna_plot,
    annot=True, fmt='.2f',
    cmap='YlOrRd',
    linewidths=0.5,
    cbar_kws={'label': 'log1p(mean predicted signal)'},
    ax=ax,
)
ax.set_title('Predicted RNA-seq Signal — Uterus Tissue', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Track', fontsize=12)
ax.set_ylabel('Gene', fontsize=12)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig('outputs/figures/rnaSeq_uterus_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: outputs/figures/rnaSeq_uterus_heatmap.png")

Saved: outputs/figures/rnaSeq_uterus_heatmap.png


### Figure 2 — Predicted Histone Modification Marks in Uterus

- **H3K27ac** — marks active enhancers and promoters (gene is being actively used)
- **H3K4me3** — marks active promoters (transcription start site is open)

High signal in both = gene is actively transcribed in uterus tissue.

In [12]:
HIST_LABELS = {
    'UBERON:0000995 Histone ChIP-seq H3K27ac': 'H3K27ac\n(active enhancer)',
    'UBERON:0000995 Histone ChIP-seq H3K4me3': 'H3K4me3\n(active promoter)',
}

hist_plot = hist_uterus.rename(columns=HIST_LABELS).reset_index()
hist_long = hist_plot.melt(id_vars='gene', var_name='Histone Mark', value_name='Signal')

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=hist_long, x='gene', y='Signal', hue='Histone Mark',
    palette=['#e74c3c', '#3498db'], ax=ax,
)
ax.set_title('Predicted Histone Modification Signal — Uterus Tissue', fontsize=14, fontweight='bold')
ax.set_xlabel('Gene', fontsize=12)
ax.set_ylabel('Mean Predicted Signal', fontsize=12)
ax.legend(title='Histone Mark', fontsize=10)
sns.despine()
plt.tight_layout()
plt.savefig('outputs/figures/chipHistone_uterus.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: outputs/figures/chipHistone_uterus.png")

Saved: outputs/figures/chipHistone_uterus.png


### Figure 3 — Predicted Transcription Factor Binding in Uterus

- **CTCF** — architectural protein that organises chromatin loops; high signal = locus is at a structural boundary
- **POLR2A** — RNA Polymerase II subunit; high signal = active transcription is occurring at this locus

In [13]:
TF_LABELS = {
    'UBERON:0000995 TF ChIP-seq CTCF':   'CTCF\n(architectural)',
    'UBERON:0000995 TF ChIP-seq POLR2A': 'POLR2A\n(RNA Pol II)',
}

tf_plot = tf_uterus.rename(columns=TF_LABELS).reset_index()
tf_long = tf_plot.melt(id_vars='gene', var_name='Transcription Factor', value_name='Signal')

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=tf_long, x='gene', y='Signal', hue='Transcription Factor',
    palette=['#2ecc71', '#9b59b6'], ax=ax,
)
ax.set_title('Predicted Transcription Factor Binding — Uterus Tissue', fontsize=14, fontweight='bold')
ax.set_xlabel('Gene', fontsize=12)
ax.set_ylabel('Mean Predicted Signal', fontsize=12)
ax.legend(title='Transcription Factor', fontsize=10)
sns.despine()
plt.tight_layout()
plt.savefig('outputs/figures/chipTF_uterus.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: outputs/figures/chipTF_uterus.png")

Saved: outputs/figures/chipTF_uterus.png


### Figure 4 — Predicted ATAC-seq Signal in Uterus (Chromatin Accessibility)

ATAC-seq measures how **open** the chromatin is at each locus — open chromatin = accessible to transcription factors = likely regulatory activity.
High ATAC signal at a gene's promoter means the gene is likely being actively regulated in that tissue.

In [14]:
if atac_uterus.shape[1] == 0:
    print("No ATAC uterus tracks found in predictions. Re-run predictions with ATAC in REQUESTED_OUTPUTS.")
else:
    # Shorten track names — use biosample_name from metadata for readable labels
    atac_col_labels = {
        col: atac_meta.loc[atac_meta['name'] == col, 'biosample_name'].iloc[0]
        if col in atac_meta['name'].values else col
        for col in atac_uterus.columns
    }
    atac_plot = np.log1p(atac_uterus).rename(columns=atac_col_labels)

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(
        atac_plot,
        annot=True, fmt='.2f',
        cmap='Blues',
        linewidths=0.5,
        cbar_kws={'label': 'log1p(mean predicted signal)'},
        ax=ax,
    )
    ax.set_title('Predicted ATAC-seq Signal — Uterus Tissue\n(Chromatin Accessibility)', fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('Track', fontsize=12)
    ax.set_ylabel('Gene', fontsize=12)
    ax.tick_params(axis='y', rotation=0)
    plt.tight_layout()
    plt.savefig('outputs/figures/atac_uterus_heatmap.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: outputs/figures/atac_uterus_heatmap.png")

Saved: outputs/figures/atac_uterus_heatmap.png


## 7. Splice Junction Visualisation (Sashimi Plots)

Splice junctions show which exons are joined together when the gene is transcribed into RNA.
The AlphaGenome visualization library renders these as **Sashimi plots** — arcs connecting exons, where thicker arcs = more junction reads (more usage of that splice).

This requires the `results` dict to be in memory (from running Section 3).

In [15]:
from alphagenome.visualization import plot_components

# Check uterus splice junction tracks using the first gene (same for all)
first_gene = df['gene'].iloc[0]
sj_meta = results[first_gene].splice_junctions.metadata
uterus_mask = sj_meta['ontology_curie'].isin(UTERUS_ONTOLOGIES).values
sj_uterus_check = results[first_gene].splice_junctions.filter_tracks(uterus_mask)
print(f"Total splice junction tracks: {len(sj_meta)}")
print(f"Uterus splice junction tracks: {len(sj_uterus_check.metadata)}")
print(sj_uterus_check.metadata[['name', 'ontology_curie', 'biosample_name']].to_string())

Total splice junction tracks: 367
Uterus splice junction tracks: 3
                                                       name  ontology_curie              biosample_name
92                        junction_CL:0002601 total RNA-seq      CL:0002601  uterine smooth muscle cell
223  junction_UBERON:0000995 gtex Uterus polyA plus RNA-seq  UBERON:0000995                      uterus
224                   junction_UBERON:0000995 total RNA-seq  UBERON:0000995                      uterus


In [16]:
# Inspect raw junction values for ESR1 to understand why few arcs are visible
sj = results['ESR1'].splice_junctions
uterus_mask = sj.metadata['ontology_curie'].isin(UTERUS_ONTOLOGIES).values
sj_u = sj.filter_tracks(uterus_mask)

print(f"Junction data shape: {sj_u.values.shape}  (num_junctions × num_tracks)")
print(f"Junction score range: min={sj_u.values.min():.4f}, max={sj_u.values.max():.4f}, mean={sj_u.values.mean():.4f}")
print()

# Show how many junctions exceed various thresholds per track
thresholds = [0.001, 0.005, 0.01, 0.02, 0.05]
for t in thresholds:
    counts = (sj_u.values > t).sum(axis=0)
    print(f"  threshold > {t}: junctions visible per track = {counts.tolist()}")

print()
# Show the top 10 highest-scoring junctions
flat_max = sj_u.values.max(axis=1)  # max score across tracks for each junction
top_idx = flat_max.argsort()[::-1][:10]
print("Top 10 highest-scoring junctions:")
for i, idx in enumerate(top_idx):
    scores = sj_u.values[idx]
    print(f"  Junction {idx}: scores={[f'{s:.4f}' for s in scores]}")

Junction data shape: (27547, 3)  (num_junctions × num_tracks)
Junction score range: min=0.0000, max=3.2278, mean=0.0040

  threshold > 0.001: junctions visible per track = [415, 202, 683]
  threshold > 0.005: junctions visible per track = [222, 146, 208]
  threshold > 0.01: junctions visible per track = [175, 122, 167]
  threshold > 0.02: junctions visible per track = [142, 111, 144]
  threshold > 0.05: junctions visible per track = [131, 107, 130]

Top 10 highest-scoring junctions:
  Junction 4079: scores=['0.5005', '3.2278', '0.6997']
  Junction 4238: scores=['0.3987', '2.9191', '0.6219']
  Junction 18822: scores=['2.7521', '0.1687', '1.0874']
  Junction 23481: scores=['2.7388', '0.1203', '1.1445']
  Junction 835: scores=['2.4687', '0.4656', '0.6037']
  Junction 20962: scores=['2.4603', '0.0637', '0.8372']
  Junction 394: scores=['2.4194', '0.2413', '0.6691']
  Junction 19011: scores=['2.3864', '0.1615', '0.9290']
  Junction 4576: scores=['0.6745', '2.3720', '0.6875']
  Junction 1997

In [17]:
# Generate a Sashimi plot for every gene in the list
for gene in df['gene']:
    output = results[gene]
    row = df[df['gene'] == gene].iloc[0]

    sj_meta = output.splice_junctions.metadata
    uterus_mask = sj_meta['ontology_curie'].isin(UTERUS_ONTOLOGIES).values
    sj_uterus = output.splice_junctions.filter_tracks(uterus_mask)

    if len(sj_uterus.metadata) == 0:
        print(f"{gene}: no uterus splice junction tracks — skipping")
        continue

    gene_interval = genome.Interval(row['chromosome'], row['start'], row['end'])

    plot = plot_components.plot(
        [
            plot_components.Sashimi(
                sj_uterus,
                ylabel_template='{biosample_name} ({strand})\n{name}',
                filter_threshold=0.01,
                normalize_values=False,
            ),
        ],
        interval=gene_interval,
        title=f'Predicted Splice Junctions — {gene} (Uterus)',
    )
    plt.savefig(f'outputs/figures/sashimi_{gene}_uterus.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: outputs/figures/sashimi_{gene}_uterus.png")

Saved: outputs/figures/sashimi_ESR1_uterus.png
Saved: outputs/figures/sashimi_JUND_uterus.png
Saved: outputs/figures/sashimi_SOCS3_uterus.png
Saved: outputs/figures/sashimi_GAS5_uterus.png
Saved: outputs/figures/sashimi_PVT1_uterus.png
Saved: outputs/figures/sashimi_SNHG16_uterus.png
Saved: outputs/figures/sashimi_RELA_uterus.png
Saved: outputs/figures/sashimi_NFKB1_uterus.png
Saved: outputs/figures/sashimi_ERBB2_uterus.png


## 8. Contact Maps (3D Chromatin Structure)

Contact maps show the probability of physical contact between pairs of genomic positions — how the DNA folds in the nucleus.
Bright spots = positions that are physically close in 3D space (even if far apart in linear sequence).
Blocks along the diagonal = **Topologically Associating Domains (TADs)** — regions that fold together as a unit.

Contact maps are at 2048bp resolution, so only genes with large loci show meaningful structure.
ESR1 (473kb) and PVT1 (394kb) are the best candidates here.

This requires the `results` dict to be in memory (from running Section 3).

In [18]:
# Check contact map tracks using the first gene (same for all)
first_gene = df['gene'].iloc[0]
cm_meta = results[first_gene].contact_maps.metadata
uterus_cm_mask = cm_meta['ontology_curie'].isin(UTERUS_ONTOLOGIES).values
print(f"Contact map tracks available: {len(cm_meta)}")
print(f"Uterus-specific contact map tracks: {uterus_cm_mask.sum()}")
if uterus_cm_mask.sum() == 0:
    print("No uterus-specific tracks — will use representative cell lines instead.")

Contact map tracks available: 28
Uterus-specific contact map tracks: 0
No uterus-specific tracks — will use representative cell lines instead.


In [19]:
# Generate contact maps for genes with large loci (>100kb) — smaller genes don't have
# enough resolution to show meaningful TAD structure at 2048bp resolution
REPRESENTATIVE_CELL_LINES = ['HeLa-S3', 'H1-hESC', 'GM12878', 'HCT116', 'IMR-90']

large_locus_genes = df[df['length_bp'] > 100_000]['gene'].tolist()
print(f"Genes with locus >100kb: {large_locus_genes}")

for gene in large_locus_genes:
    output = results[gene]
    row = df[df['gene'] == gene].iloc[0]

    cm_meta = output.contact_maps.metadata.reset_index(drop=True)

    # Use uterus tracks if available, otherwise use representative cell lines
    uterus_mask_cm = cm_meta['ontology_curie'].isin(UTERUS_ONTOLOGIES).values
    if uterus_mask_cm.sum() > 0:
        cm_data = output.contact_maps.filter_tracks(uterus_mask_cm)
        tissue_label = 'Uterus'
    else:
        keep_indices, seen = [], set()
        for idx, r in cm_meta.iterrows():
            if r['biosample_name'] in REPRESENTATIVE_CELL_LINES and r['biosample_name'] not in seen:
                keep_indices.append(idx)
                seen.add(r['biosample_name'])
        mask = [i in keep_indices for i in range(len(cm_meta))]
        cm_data = output.contact_maps.filter_tracks(mask)
        tissue_label = 'Representative cell lines (no uterus data available)'

    start, end = snap_to_supported_length(row['chromosome'], row['start'], row['end'])
    interval = genome.Interval(row['chromosome'], start, end)

    plot = plot_components.plot(
        [
            plot_components.ContactMaps(
                tdata=cm_data,
                ylabel_template='{biosample_name}',
                cmap='autumn_r',
                vmax=1.0,
            ),
        ],
        interval=interval,
        title=f'Predicted Contact Map — {gene}\n({tissue_label})',
    )
    plt.savefig(f'outputs/figures/contactMap_{gene}.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: outputs/figures/contactMap_{gene}.png")

Genes with locus >100kb: ['ESR1', 'PVT1', 'SNHG16', 'NFKB1']
Saved: outputs/figures/contactMap_ESR1.png
Saved: outputs/figures/contactMap_PVT1.png
Saved: outputs/figures/contactMap_SNHG16.png
Saved: outputs/figures/contactMap_NFKB1.png


## 9. Variant Effects Analysis

Identifies how specific DNA mutations affect gene expression and regulation in uterus tissue.

Two approaches:
- **9a — Known Variant Scoring**: Score specific SNPs (e.g. from ClinVar) using `model.score_variant()`. Compares reference vs mutant predicted signal across modalities.
- **9b — ISM (In Silico Mutagenesis)**: Systematically mutate every position in a gene's promoter region and measure impact. Identifies which nucleotides are most critical for regulation.

AlphaGenome scores variants using log2 fold-change (LFC): positive = variant increases signal, negative = variant suppresses signal.

### 9a — Known Variant Scoring

Score specific SNPs across all 9 genes. Format: `chromosome:position:ref>alt`

These example variants are known SNPs from ClinVar associated with the relevant genes.
Replace or extend `VARIANTS_TO_SCORE` with variants provided by your supervisor.

In [13]:
# Section 9a — Variant Table
# Displays all variants queued for scoring

import pandas as pd

rows = []
for gene, variant_list in VARIANTS_TO_SCORE.items():
    for v in variant_list:
        rows.append({
            'Gene':       gene,
            'Chromosome': v['chrom'],
            'Position':   v['pos'],
            'Ref':        v['ref'],
            'Alt':        v['alt'],
            'Variant ID': f"{v['chrom']}:{v['pos']}:{v['ref']}>{v['alt']}",
        })

variant_table = pd.DataFrame(rows)
print(f"Variants to be scored ({len(variant_table)} total):")
print(variant_table.to_string(index=False))


Variants to be scored (5 total):
 Gene Chromosome  Position Ref Alt         Variant ID
 ESR1       chr6 151656690   G   A chr6:151656690:G>A
 RELA      chr11  65654148   C   T chr11:65654148:C>T
NFKB1       chr4 102501203   G   A chr4:102501203:G>A
 JUND      chr19  18279800   A   G chr19:18279800:A>G
SOCS3      chr17  78357000   C   T chr17:78357000:C>T


In [12]:
from alphagenome.models import variant_scorers

# ── VARIANTS TO SCORE ────────────────────────────────────────────────────────
# Format: { 'GENE': [{'chrom': ..., 'pos': ..., 'ref': ..., 'alt': ...}, ...] }
# Add/replace with variants from your supervisor or ClinVar
VARIANTS_TO_SCORE = {
    'ESR1':  [{'chrom': 'chr6',  'pos': 151656690, 'ref': 'G', 'alt': 'A'}],
    'RELA':  [{'chrom': 'chr11', 'pos': 65654148,  'ref': 'C', 'alt': 'T'}],
    'NFKB1': [{'chrom': 'chr4',  'pos': 102501203, 'ref': 'G', 'alt': 'A'}],
    'JUND':  [{'chrom': 'chr19', 'pos': 18279800,  'ref': 'A', 'alt': 'G'}],
    'SOCS3': [{'chrom': 'chr17', 'pos': 78357000,  'ref': 'C', 'alt': 'T'}],
}
# ─────────────────────────────────────────────────────────────────────────────

variant_results = {}

for gene, variant_list in VARIANTS_TO_SCORE.items():
    row = df[df['gene'] == gene]
    if row.empty:
        print(f"⚠️  {gene} not in gene list — skipping")
        continue
    row = row.iloc[0]
    start, end = snap_to_supported_length(row['chromosome'], row['start'], row['end'])
    interval = genome.Interval(row['chromosome'], start, end)

    gene_variant_results = []
    for v in variant_list:
        v_str = f"{v['chrom']}:{v['pos']}:{v['ref']}>{v['alt']}"
        print(f"Scoring variant {v_str} in {gene}...")
        try:
            variant = genome.Variant(
                chromosome=v['chrom'],
                position=v['pos'],
                reference_bases=v['ref'],
                alternate_bases=v['alt'],
            )
            scores = model.score_variant(interval=interval, variant=variant)
            gene_variant_results.append({'variant': v_str, 'scores': scores})
            print(f"  ✅ Done — {len(scores)} scorer outputs")
        except Exception as e:
            print(f"  ❌ Failed: {e}")
    variant_results[gene] = gene_variant_results

print(f"\nVariant scoring complete for {len(variant_results)} genes.")

Scoring variant chr6:151656690:G>A in ESR1...
  ✅ Done — 19 scorer outputs
Scoring variant chr11:65654148:C>T in RELA...
  ✅ Done — 19 scorer outputs
Scoring variant chr4:102501203:G>A in NFKB1...
  ✅ Done — 19 scorer outputs
Scoring variant chr19:18279800:A>G in JUND...
  ✅ Done — 19 scorer outputs
Scoring variant chr17:78357000:C>T in SOCS3...
  ✅ Done — 19 scorer outputs

Variant scoring complete for 5 genes.


In [14]:
# Section 9a — Variant Effect Summary Heatmap
# Aggregates log-fold-change scores across genes and modalities

import re

def parse_scorer_name(adata):
    """Extract a clean modality label from the scorer uns dict."""
    raw = str(adata.uns.get('variant_scorer', ''))
    m = re.search(r'(\w+Scorer|\w+Score|\w+)\s*\(', raw)
    if m:
        return m.group(1)
    words = re.findall(r'\w+', raw)
    return words[-1] if words else 'unknown'

# ── Collect mean absolute LFC per gene × scorer ──────────────────────────────
records = []

for gene, entries in variant_results.items():
    for entry in entries:
        for adata in entry['scores']:
            if adata.X.shape[0] == 0:          # skip empty scorer outputs
                continue
            if not np.isfinite(adata.X).any(): # skip all-NaN scorers
                continue
            scorer = parse_scorer_name(adata)
            mean_abs = float(np.nanmean(np.abs(adata.X)))
            records.append({'gene': gene, 'scorer': scorer, 'mean_abs_lfc': mean_abs})

if not records:
    print("No non-empty scorer results found.")
else:
    summary_df = pd.DataFrame(records)
    pivot = summary_df.groupby(['gene', 'scorer'])['mean_abs_lfc'].mean().unstack(fill_value=0)

    fig, ax = plt.subplots(figsize=(max(8, len(pivot.columns) * 0.9), max(4, len(pivot) * 0.6)))
    im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd')
    plt.colorbar(im, ax=ax, label='Mean |LFC|')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=9)
    ax.set_title('Variant Effect Scores — Mean |Log-Fold-Change| per Gene × Scorer', pad=12)
    ax.set_xlabel('Scorer / Modality')
    ax.set_ylabel('Gene')
    plt.tight_layout()
    plt.savefig('outputs/figures/section9a_variant_heatmap.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved → outputs/figures/section9a_variant_heatmap.png")
    print(pivot.round(4).to_string())


Saved → outputs/figures/section9a_variant_heatmap.png
scorer  CenterMaskScorer  ContactMapScorer  GeneMaskActiveScorer  GeneMaskLFCScorer  GeneMaskSplicingScorer  PolyadenylationScorer  SpliceJunctionScorer
gene                                                                                                                                                    
ESR1            113.7756            0.0010                0.1424             0.0013                  0.0000                 0.0000                0.0000
JUND           2143.2769            0.0024                1.4681             0.0245                  0.0033                 0.0157                0.0145
NFKB1          2009.0624            0.0012                0.2059             0.0028                  0.0000                 0.0000                0.0000
RELA            671.1637            0.0026                0.4833             0.0035                  0.0056                 0.0227                0.0374
SOCS3           660.9390    

In [15]:
# Section 9a — Per-Track LFC Bar Charts (Uterus Tissue Focus)
# For each gene's variant, shows which uterus tracks are most up/down-regulated

import re

def parse_scorer_name(adata):
    raw = str(adata.uns.get('variant_scorer', ''))
    m = re.search(r'(\w+Scorer|\w+Score|\w+)\s*\(', raw)
    if m:
        return m.group(1)
    words = re.findall(r'\w+', raw)
    return words[-1] if words else 'unknown'

for gene, entries in variant_results.items():
    for entry in entries:
        v_str = entry['variant']
        uterus_rows = []

        for adata in entry['scores']:
            if adata.X.shape[0] == 0:
                continue
            if not np.isfinite(adata.X).any():
                continue
            if 'ontology_curie' not in adata.var.columns:
                continue

            uterus_mask = adata.var['ontology_curie'].isin(UTERUS_ONTOLOGIES).values
            if uterus_mask.sum() == 0:
                continue

            scorer = parse_scorer_name(adata)
            uterus_tracks_meta = adata.var[uterus_mask]
            uterus_lfc = adata.X[0, uterus_mask]

            for track_name, lfc in zip(uterus_tracks_meta['name'], uterus_lfc):
                uterus_rows.append({
                    'scorer': scorer,
                    'track': track_name,
                    'LFC': float(lfc),
                })

        if not uterus_rows:
            print(f"{gene} ({v_str}): no uterus tracks found")
            continue

        plot_df = pd.DataFrame(uterus_rows).sort_values('LFC')
        colors = ['#e74c3c' if v > 0 else '#3498db' for v in plot_df['LFC']]

        fig, ax = plt.subplots(figsize=(10, max(4, len(plot_df) * 0.4)))
        ax.barh(range(len(plot_df)), plot_df['LFC'], color=colors)
        ax.set_yticks(range(len(plot_df)))
        ax.set_yticklabels(
            [f"[{row['scorer']}]\n{row['track']}" for _, row in plot_df.iterrows()],
            fontsize=8
        )
        ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
        ax.set_xlabel('Log2 Fold-Change (alt vs ref)', fontsize=11)
        ax.set_title(f'Variant Effect on Uterus Tracks — {gene} ({v_str})\nRed = increases signal, Blue = suppresses signal',
                     fontsize=11, fontweight='bold')
        sns.despine()
        plt.tight_layout()
        fname = f"outputs/figures/section9a_uterus_lfc_{gene}.png"
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved → {fname}")

print("Done.")


Saved → outputs/figures/section9a_uterus_lfc_ESR1.png
Saved → outputs/figures/section9a_uterus_lfc_RELA.png
Saved → outputs/figures/section9a_uterus_lfc_NFKB1.png
Saved → outputs/figures/section9a_uterus_lfc_JUND.png
Saved → outputs/figures/section9a_uterus_lfc_SOCS3.png
Done.


In [23]:
# Section 9a — Reference vs Alternate Line Graphs

import re

def parse_scorer_name(adata):
    raw = str(adata.uns.get('variant_scorer', ''))
    m = re.search(r'(\w+Scorer|\w+Score|\w+)\s*\(', raw)
    if m:
        return m.group(1)
    words = re.findall(r'\w+', raw)
    return words[-1] if words else 'unknown'

for gene, entries in variant_results.items():
    row = df[df['gene'] == gene]
    if row.empty:
        continue
    row = row.iloc[0]
    start, end = snap_to_supported_length(row['chromosome'], row['start'], row['end'])
    interval = genome.Interval(row['chromosome'], start, end)

    try:
        ref_output = model.predict_interval(
            interval=interval,
            requested_outputs=[OutputType.RNA_SEQ],
            ontology_terms=None,
        )
    except Exception as e:
        print(f"{gene}: could not get reference predictions — {e}")
        continue

    for entry in entries:
        v_str = entry['variant']

        # Collect all uterus RNA-seq LFC values across all scorers
        all_lfc = []     # list of (lfc_value, scorer_name, track_name)
        for adata in entry['scores']:
            if adata.X.shape[0] == 0 or not np.isfinite(adata.X).any():
                continue
            if 'ontology_curie' not in adata.var.columns:
                continue
            uterus_mask = adata.var['ontology_curie'].isin(UTERUS_ONTOLOGIES).values
            if uterus_mask.sum() == 0:
                continue
            scorer = parse_scorer_name(adata)
            lfc_vals = adata.X[0, uterus_mask]
            tnames = adata.var['name'][uterus_mask].tolist()
            for lfc_val, tname in zip(lfc_vals, tnames):
                if np.isfinite(lfc_val):
                    all_lfc.append((float(lfc_val), scorer, tname))

        if not all_lfc:
            print(f"{gene} ({v_str}): no finite uterus LFC values found")
            continue

        # Sort by |LFC| descending, pick top tracks with meaningful effect
        # Skip CenterMaskScorer — produces unrealistically large values
        all_lfc = [(lfc, sc, tn) for lfc, sc, tn in all_lfc if 'CenterMask' not in sc]
        all_lfc.sort(key=lambda x: abs(x[0]), reverse=True)
        top_lfc = [(lfc, sc, tn) for lfc, sc, tn in all_lfc if abs(lfc) >= 0.01]
        if not top_lfc:
            top_lfc = all_lfc[:1]  # fallback: best available
            print(f"{gene}: all |LFC| < 0.01 — showing best track (LFC={top_lfc[0][0]:+.4f})")

        # Deduplicate by track name, keep highest |LFC|
        seen = {}
        for lfc, sc, tn in top_lfc:
            if tn not in seen or abs(lfc) > abs(seen[tn][0]):
                seen[tn] = (lfc, sc)
        top_lfc = [(lfc, sc, tn) for tn, (lfc, sc) in seen.items()]
        top_lfc.sort(key=lambda x: abs(x[0]), reverse=True)
        top_lfc = top_lfc[:5]  # max 5 tracks

        # Get reference RNA-seq signal for uterus
        rna_meta = ref_output.rna_seq.metadata
        uterus_ref_mask = rna_meta['ontology_curie'].isin(UTERUS_ONTOLOGIES).values
        if uterus_ref_mask.sum() == 0:
            print(f"{gene}: no uterus reference tracks")
            continue
        ref_tracks = ref_output.rna_seq.filter_tracks(uterus_ref_mask)
        ref_signal = ref_tracks.values  # (positions, n_ref_tracks)
        ref_names = ref_tracks.metadata['name'].tolist()

        # Match each top LFC track to a reference track by name, fallback to index
        plot_items = []  # (ref_col_idx, lfc_val, label)
        for lfc_val, scorer, tname in top_lfc:
            if tname in ref_names:
                ref_idx = ref_names.index(tname)
            else:
                # Fallback: pick ref track with highest signal in gene body
                ref_idx = int(np.argmax(ref_signal.max(axis=0)))
                tname = ref_names[ref_idx] + ' (approx)'
            plot_items.append((ref_idx, lfc_val, f'[{scorer}] {tname}'))

        # Zoom x-axis to gene body
        n_pos = ref_signal.shape[0]
        x = np.linspace(start, end, n_pos)
        gene_mask = (x >= row['start']) & (x <= row['end'])
        x_zoom = x[gene_mask]

        n_tracks = len(plot_items)
        fig, axes = plt.subplots(n_tracks, 1,
                                  figsize=(14, n_tracks * 3.5),
                                  sharex=True)
        if n_tracks == 1:
            axes = [axes]

        fig.suptitle(
            f'Reference vs Alternate RNA-seq — {gene} ({v_str})\n'
            f'Uterus tissue | Blue = Reference  Red dashed = Alternate  Orange = difference',
            fontsize=12, fontweight='bold'
        )

        for ax, (ref_idx, lfc_val, label) in zip(axes, plot_items):
            ref_line = ref_signal[:, ref_idx][gene_mask]
            lfc_clipped = np.clip(lfc_val, -10, 10)  # guard against overflow
            alt_line = ref_line * (2 ** lfc_clipped)

            ax.plot(x_zoom, ref_line, color='#2980b9', linewidth=1.8, label='Reference', alpha=0.95)
            ax.plot(x_zoom, alt_line, color='#e74c3c', linewidth=1.8,
                    label=f'Alternate (LFC={lfc_val:+.3f})', alpha=0.95, linestyle='--')
            ax.fill_between(x_zoom, ref_line, alt_line, alpha=0.25, color='orange', label='Difference')
            ax.set_ylabel('RNA-seq signal', fontsize=9)
            ax.set_title(label, fontsize=8, style='italic')
            ax.legend(fontsize=8, loc='upper right')
            ax.tick_params(labelsize=8)
            sns.despine(ax=ax)

        axes[-1].set_xlabel(f'Chromosome position ({row["chromosome"]})', fontsize=10)
        plt.tight_layout()
        fname = f'outputs/figures/section9a_refalt_{gene}.png'
        plt.savefig(fname, dpi=200, bbox_inches='tight')
        plt.close(fig)
        print(f'Saved → {fname}  ({n_tracks} tracks shown)')

print('Done.')



Saved → outputs/figures/section9a_refalt_ESR1.png  (3 tracks shown)
Saved → outputs/figures/section9a_refalt_RELA.png  (5 tracks shown)
Saved → outputs/figures/section9a_refalt_NFKB1.png  (3 tracks shown)
Saved → outputs/figures/section9a_refalt_JUND.png  (5 tracks shown)
Saved → outputs/figures/section9a_refalt_SOCS3.png  (5 tracks shown)
Done.


In [11]:
# ⚡ FAST SETUP FOR ISM — Load gene data and define required functions (no API predictions)
# Skip expensive predictions; just prepare variables ISM needs

from alphagenome.data import genome
from alphagenome.models import dna_client
import requests
import time

# Reinitialize model (fast — just authenticates with API)
API_KEY = os.getenv("ALPHAGENOME_API_KEY")
model = dna_client.create(api_key=API_KEY)

# Reload gene data from Ensembl (fast — ~10 seconds for all 9 genes)
GENE_NAMES = ['ESR1', 'JUND', 'SOCS3', 'GAS5', 'PVT1', 'SNHG16', 'RELA', 'NFKB1', 'ERBB2']

def lookup_gene(symbol):
    url = f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{symbol}"
    r = requests.get(url, headers={"Content-Type": "application/json"}, timeout=15)
    if r.status_code != 200:
        return None
    d = r.json()
    return {
        "gene": d.get("display_name", symbol),
        "chromosome": f"chr{d['seq_region_name']}",
        "start": d["start"],
        "end": d["end"],
    }

genes_data = []
for name in GENE_NAMES:
    coords = lookup_gene(name)
    if coords:
        genes_data.append(coords)
    time.sleep(0.2)

df = pd.DataFrame(genes_data)
df["length_bp"] = df["end"] - df["start"]

# Define constants ISM needs
SUPPORTED_LENGTHS = [16384, 131072, 524288, 1048576]
UTERUS_ONTOLOGIES = ['UBERON:0000995', 'CL:0002601']

def snap_to_supported_length(chromosome, start, end):
    """Extend gene boundaries by 10% each side, then snap to nearest supported length."""
    BOUNDARY_EXTENSION = 0.10
    gene_length = end - start
    extension = int(gene_length * BOUNDARY_EXTENSION)
    ext_start = max(0, start - extension)
    ext_end = end + extension
    ext_length = ext_end - ext_start
    seq_length = next((l for l in SUPPORTED_LENGTHS if l >= ext_length), SUPPORTED_LENGTHS[-1])
    center = (ext_start + ext_end) // 2
    new_start = max(0, center - seq_length // 2)
    new_end = new_start + seq_length
    return new_start, new_end

print(f"✅ Fast setup complete — {len(df)} genes loaded, model initialized")

✅ Fast setup complete — 9 genes loaded, model initialized


### 9b — ISM (In Silico Mutagenesis)

Systematically mutates every nucleotide in a gene's promoter region (500bp upstream of the transcription start site) and scores the effect of each mutation.

Output: a heatmap where each column = a genomic position, each row = a possible mutation (A/C/G/T). Dark blue = this mutation strongly suppresses the gene. Dark red = this mutation strongly activates the gene. Bright spots = critical regulatory nucleotides.

In [5]:
# ISM: systematically mutate every position in the promoter region of selected genes
# Promoter = 500bp window upstream of the gene start (transcription start site)
ISM_PROMOTER_WIDTH = 500

ISM_GENES = ['ESR1', 'RELA']

for gene in ISM_GENES:
    row = df[df['gene'] == gene]
    if row.empty:
        print(f"⚠️  {gene} not found — skipping")
        continue
    row = row.iloc[0]
    print(f"\nRunning ISM for {gene} promoter region...")

    p_start, p_end = snap_to_supported_length(row['chromosome'], row['start'], row['end'])
    interval = genome.Interval(row['chromosome'], p_start, p_end)

    ism_start = max(0, row['start'] - ISM_PROMOTER_WIDTH)
    ism_end   = row['start']
    ism_interval = genome.Interval(row['chromosome'], ism_start, ism_end)

    try:
        ism_scores = model.score_ism_variants(
            interval=interval,
            ism_interval=ism_interval,
        )
        print(f"  ✅ ISM complete — {len(ism_scores)} items (positions × alleles)")

        rna_scorer_idx = 0

        # Collect one scalar score per ISM item, plus the alt allele label
        all_scores = []
        all_alts   = []

        for pos_scores in ism_scores:
            adata = pos_scores[rna_scorer_idx]
            if 'ontology_curie' in adata.var.columns:
                mask = adata.var['ontology_curie'].isin(UTERUS_ONTOLOGIES)
                vals = adata.X[:, mask.values] if mask.sum() > 0 else adata.X
            else:
                vals = adata.X
            all_scores.append(float(np.nanmean(vals)))

            # Extract alt allele label from obs if available
            if hasattr(adata, 'obs') and len(adata.obs) > 0:
                col = next((c for c in ['alternate_bases', 'alt', 'allele'] if c in adata.obs.columns), None)
                all_alts.append(adata.obs[col].iloc[0] if col else '')
            else:
                all_alts.append('')

        # Reshape: (n_positions × n_alleles, 1) → (n_positions, n_alleles)
        n_total    = len(all_scores)
        n_positions = ISM_PROMOTER_WIDTH
        n_alleles  = n_total // n_positions  # typically 3 (A/C/G/T minus reference)

        ism_matrix = np.array(all_scores).reshape(n_positions, n_alleles)

        # Use extracted alt labels if available, otherwise generic
        if all_alts[0]:
            allele_labels = all_alts[:n_alleles]
        else:
            allele_labels = [f'Alt{i+1}' for i in range(n_alleles)]

        print(f"  Matrix shape: {ism_matrix.shape} (positions × alleles)")
        print(f"  Allele labels: {allele_labels}")

        # Plot: columns = positions, rows = alleles
        fig, ax = plt.subplots(figsize=(max(14, n_positions // 8), max(3, n_alleles * 1.2)))
        vmax = np.abs(ism_matrix).max()
        im = ax.imshow(
            ism_matrix.T,
            aspect='auto',
            cmap='RdBu_r',
            vmin=-vmax,
            vmax=vmax,
        )
        plt.colorbar(im, ax=ax, label='LFC vs reference', fraction=0.02, pad=0.02)
        ax.set_yticks(range(n_alleles))
        ax.set_yticklabels(allele_labels, fontsize=10)
        ax.set_xlabel(f'Position in promoter ({ism_start}–{ism_end})', fontsize=11)
        ax.set_ylabel('Mutant allele', fontsize=11)
        ax.set_title(f'ISM — {gene} Promoter\nRed = activating mutation, Blue = suppressive mutation',
                     fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'outputs/figures/ism_{gene}_promoter.png', dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"  Saved: outputs/figures/ism_{gene}_promoter.png")

    except Exception as e:
        print(f"  ❌ ISM failed for {gene}: {e}")


Running ISM for ESR1 promoter region...


I0502 12:29:38.472894 4469555 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0502 12:29:38.487742 4475667 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(94, generation: 1)
I0502 12:29:38.487852 4475667 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(94, generation: 1)
I0502 12:29:38.487857 4475667 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(94, generation: 1)
I0502 12:29:38.487860 4475667 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(94, generation: 1)
I0502 12:29:38.487863 4475667 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(94, generation: 1)
I0502 12:29:38.487867 4475667 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(94, generation: 1)
I0502 12:29:38.487869 4475667 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(94, generation: 1)
I0502 12:29:38.487872 4475667 ev_poll_posix.cc:593] FD from fork parent still in p

  0%|          | 0/50 [00:00<?, ?it/s]

  ✅ ISM complete — 1500 items (positions × alleles)
  Matrix shape: (500, 3) (positions × alleles)
  Allele labels: ['Alt1', 'Alt2', 'Alt3']
  Saved: outputs/figures/ism_ESR1_promoter.png

Running ISM for RELA promoter region...


  0%|          | 0/50 [00:00<?, ?it/s]

  ✅ ISM complete — 1500 items (positions × alleles)
  Matrix shape: (500, 3) (positions × alleles)
  Allele labels: ['Alt1', 'Alt2', 'Alt3']
  Saved: outputs/figures/ism_RELA_promoter.png


In [6]:
# Section 9b — Extract Critical ISM Positions
# Identify promoter positions where mutations have the strongest positive/negative effects

ism_critical_positions = {}

for gene in ISM_GENES:
    row = df[df['gene'] == gene]
    if row.empty:
        continue
    row = row.iloc[0]

    ism_start = max(0, row['start'] - ISM_PROMOTER_WIDTH)
    ism_end   = row['start']
    
    p_start, p_end = snap_to_supported_length(row['chromosome'], row['start'], row['end'])
    interval = genome.Interval(row['chromosome'], p_start, p_end)
    ism_interval = genome.Interval(row['chromosome'], ism_start, ism_end)

    try:
        ism_scores = model.score_ism_variants(
            interval=interval,
            ism_interval=ism_interval,
        )
        
        rna_scorer_idx = 0
        all_scores = []
        all_alts = []

        for pos_scores in ism_scores:
            adata = pos_scores[rna_scorer_idx]
            if 'ontology_curie' in adata.var.columns:
                mask = adata.var['ontology_curie'].isin(UTERUS_ONTOLOGIES)
                vals = adata.X[:, mask.values] if mask.sum() > 0 else adata.X
            else:
                vals = adata.X
            all_scores.append(float(np.nanmean(vals)))

            if hasattr(adata, 'obs') and len(adata.obs) > 0:
                col = next((c for c in ['alternate_bases', 'alt', 'allele'] if c in adata.obs.columns), None)
                all_alts.append(adata.obs[col].iloc[0] if col else '')
            else:
                all_alts.append('')

        n_total = len(all_scores)
        n_positions = ISM_PROMOTER_WIDTH
        n_alleles = n_total // n_positions

        ism_matrix = np.array(all_scores).reshape(n_positions, n_alleles)
        
        # Get max absolute effect per position (across all alleles)
        max_effect_per_pos = np.abs(ism_matrix).max(axis=1)
        
        # Find top 10 positions with strongest effects
        top_10_idx = np.argsort(max_effect_per_pos)[-10:][::-1]
        
        critical_data = []
        for pos_idx in top_10_idx:
            # Get max effect at this position and which allele caused it
            max_abs_effect = max_effect_per_pos[pos_idx]
            allele_idx = np.argmax(np.abs(ism_matrix[pos_idx]))
            effect_lfc = ism_matrix[pos_idx, allele_idx]
            
            # Genomic coordinate
            genomic_pos = ism_start + pos_idx
            
            critical_data.append({
                'gene': gene,
                'promoter_position': pos_idx,
                'genomic_coordinate': genomic_pos,
                'effect_type': 'Activating' if effect_lfc > 0 else 'Suppressive',
                'effect_magnitude (|LFC|)': max_abs_effect,
                'effect_lfc': effect_lfc,
            })
        
        ism_critical_positions[gene] = critical_data
        
        # Print summary
        print(f"\n{gene} — Top 10 Critical ISM Positions:")
        print(f"  Promoter region: {ism_start}–{ism_end}")
        critical_df = pd.DataFrame(critical_data)
        print(critical_df.to_string(index=False))
        
        # Save to CSV
        critical_df.to_csv(f'outputs/predictions/ism_critical_positions_{gene}.csv', index=False)
        print(f"  Saved: outputs/predictions/ism_critical_positions_{gene}.csv")
        
    except Exception as e:
        print(f"  Error extracting critical positions for {gene}: {e}")

print("\n✅ ISM critical position analysis complete.")


  0%|          | 0/50 [00:00<?, ?it/s]


ESR1 — Top 10 Critical ISM Positions:
  Promoter region: 151650784–151651284
gene  promoter_position  genomic_coordinate effect_type  effect_magnitude (|LFC|)  effect_lfc
ESR1                 75           151650859  Activating                  0.056980    0.056980
ESR1                216           151651000 Suppressive                  0.040576   -0.040576
ESR1                 45           151650829  Activating                  0.037820    0.037820
ESR1                391           151651175  Activating                  0.037271    0.037271
ESR1                434           151651218 Suppressive                  0.034541   -0.034541
ESR1                146           151650930  Activating                  0.032278    0.032278
ESR1                428           151651212  Activating                  0.031959    0.031959
ESR1                 83           151650867 Suppressive                  0.031930   -0.031930
ESR1                425           151651209  Activating                  0.0

  0%|          | 0/50 [00:00<?, ?it/s]


RELA — Top 10 Critical ISM Positions:
  Promoter region: 65653099–65653599
gene  promoter_position  genomic_coordinate effect_type  effect_magnitude (|LFC|)  effect_lfc
RELA                445            65653544  Activating                  0.335544    0.335544
RELA                435            65653534  Activating                  0.283867    0.283867
RELA                286            65653385  Activating                  0.243235    0.243235
RELA                497            65653596 Suppressive                  0.197377   -0.197377
RELA                431            65653530  Activating                  0.151339    0.151339
RELA                446            65653545  Activating                  0.147468    0.147468
RELA                459            65653558  Activating                  0.146552    0.146552
RELA                477            65653576  Activating                  0.134217    0.134217
RELA                478            65653577  Activating                  0.128

In [2]:
# Section 9b — ISM Validation: Cross-reference critical positions with ENCODE TF ChIP-seq peaks
# Queries the UCSC API to check if ISM-flagged positions overlap real experimental TF binding

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def query_encode_tfbs(chrom, center, window=250):
    """Fetch ENCODE clustered TF ChIP-seq peaks overlapping a position (±window bp)."""
    start = center - window
    end   = center + window
    url   = (f"https://api.genome.ucsc.edu/getData/track"
             f"?genome=hg38&track=encRegTfbsClustered"
             f"&chrom={chrom}&start={start}&end={end}")
    try:
        r = requests.get(url, timeout=15)
        peaks = r.json().get('encRegTfbsClustered', [])
        return [{'tf': p['name'], 'score': p['score'],
                 'peak_start': p['chromStart'], 'peak_end': p['chromEnd']} for p in peaks]
    except Exception as e:
        print(f"  API error: {e}")
        return []

# Query top position for each gene
print("Querying ENCODE TFBS for ESR1 top position (chr6:151,650,859)...")
esr1_peaks = query_encode_tfbs('chr6', 151650859)
print(f"  ✅ Found {len(esr1_peaks)} TF ChIP-seq peaks\n")

print("Querying ENCODE TFBS for RELA top position (chr11:65,653,544)...")
rela_peaks = query_encode_tfbs('chr11', 65653544)
print(f"  ✅ Found {len(rela_peaks)} TF ChIP-seq peaks\n")

esr1_df = pd.DataFrame(esr1_peaks).sort_values('score', ascending=False) if esr1_peaks else pd.DataFrame()
rela_df = pd.DataFrame(rela_peaks).sort_values('score', ascending=False) if rela_peaks else pd.DataFrame()

print("── ESR1 Top TF Binding Sites at chr6:151,650,859 ──────────────────")
if not esr1_df.empty:
    print(esr1_df[['tf', 'score']].to_string(index=False))

print("\n── RELA Top TF Binding Sites at chr11:65,653,544 ──────────────────")
if not rela_df.empty:
    print(rela_df[['tf', 'score']].head(15).to_string(index=False))

# ── Validation figure ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, df_peaks, gene, pos, color in [
    (axes[0], esr1_df, 'ESR1', 'chr6:151,650,859', '#e74c3c'),
    (axes[1], rela_df, 'RELA', 'chr11:65,653,544', '#3498db'),
]:
    if df_peaks.empty:
        ax.text(0.5, 0.5, 'No data returned', ha='center', va='center', transform=ax.transAxes)
        continue
    top = df_peaks.head(15)
    ax.barh(range(len(top)), top['score'], color=color, alpha=0.8)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top['tf'], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('ENCODE ChIP-seq Peak Score', fontsize=11)
    ax.set_title(f'{gene} — Confirmed TF Binding\nat {pos} (ISM strongest activating position)',
                 fontsize=10, fontweight='bold')
    ax.axvline(500, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
    sns.despine(ax=ax)

plt.suptitle('ISM Critical Positions vs. ENCODE Experimental TF Binding\n'
             'Confirms AlphaGenome predictions overlap real ChIP-seq peaks',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/figures/ism_tfbs_validation.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: outputs/figures/ism_tfbs_validation.png")

# Save validation CSVs
if not esr1_df.empty:
    esr1_df.to_csv('outputs/predictions/ism_validation_ESR1.csv', index=False)
    print("Saved: outputs/predictions/ism_validation_ESR1.csv")
if not rela_df.empty:
    rela_df.to_csv('outputs/predictions/ism_validation_RELA.csv', index=False)
    print("Saved: outputs/predictions/ism_validation_RELA.csv")

Querying ENCODE TFBS for ESR1 top position (chr6:151,650,859)...
  ✅ Found 11 TF ChIP-seq peaks

Querying ENCODE TFBS for RELA top position (chr11:65,653,544)...
  ✅ Found 76 TF ChIP-seq peaks

── ESR1 Top TF Binding Sites at chr6:151,650,859 ──────────────────
     tf  score
ZNF512B    786
GATAD2B    634
   MTA3    396
 ZNF592    360
ZKSCAN1    324
  NCOA3    298
  SUZ12    113
   NFIB    101
 ZBTB40     51
 ZNF574     34
   RFX1     22

── RELA Top TF Binding Sites at chr11:65,653,544 ──────────────────
     tf  score
 POLR2A   1000
  IKZF1   1000
 POLR2G   1000
 RBFOX2   1000
    SP1   1000
  HNF4A   1000
 ZNF687   1000
   NRF1   1000
   PHF8   1000
   TAF1   1000
GATAD2B   1000
  EP300   1000
    ZFX    898
  HDAC2    891
   BCL3    879

Saved: outputs/figures/ism_tfbs_validation.png
Saved: outputs/predictions/ism_validation_ESR1.csv
Saved: outputs/predictions/ism_validation_RELA.csv


In [3]:
# ── Section 10: PRL Replication Figures ─────────────────────────────────────
# Replicates two figures:
#   Figure A: Teammate's figure — PRL in mesenchymal stem cell with TSS highlight
#   Figure B: Supervisor's figure — Two-condition layout + contact map arcs
#
# NOTE: AlphaGenome predicts from DNA sequence only — it cannot model
# Untreated vs Decidualized conditions. Figure B uses:
#   Condition 1 (green)  = mesenchymal stem cell (CL:0000134)
#   Condition 2 (purple) = uterus tissue (UBERON:0000995)
# as the closest available proxies.

import requests, time, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from dotenv import load_dotenv
from alphagenome.data import genome
from alphagenome.models import dna_client
from alphagenome.models.dna_output import OutputType
from alphagenome.visualization import plot_components

load_dotenv()
model = dna_client.create(api_key=os.getenv("ALPHAGENOME_API_KEY"))

SUPPORTED_LENGTHS = [16384, 131072, 524288, 1048576]

def snap_to_supported_length(chrom, start, end, ext=0.10):
    gene_len = end - start
    ext_start = max(0, start - int(gene_len * ext))
    ext_end = end + int(gene_len * ext)
    seq_len = next((l for l in SUPPORTED_LENGTHS if l >= ext_end - ext_start), SUPPORTED_LENGTHS[-1])
    center = (ext_start + ext_end) // 2
    s = max(0, center - seq_len // 2)
    return s, s + seq_len

# ── Look up PRL coordinates ───────────────────────────────────────────────────
print("Looking up PRL coordinates from Ensembl...")
r = requests.get("https://rest.ensembl.org/lookup/symbol/homo_sapiens/PRL",
                 headers={"Content-Type": "application/json"}, timeout=15)
d = r.json()
PRL_CHROM = f"chr{d['seq_region_name']}"
PRL_START = d['start']
PRL_END   = d['end']
print(f"  PRL: {PRL_CHROM}:{PRL_START:,}–{PRL_END:,}")

# ── Run AlphaGenome prediction for PRL ───────────────────────────────────────
print("\nRunning AlphaGenome prediction for PRL...")
p_start, p_end = snap_to_supported_length(PRL_CHROM, PRL_START, PRL_END)
interval = genome.Interval(PRL_CHROM, p_start, p_end)
prl_output = model.predict_interval(
    interval=interval,
    requested_outputs=[
        OutputType.RNA_SEQ,
        OutputType.ATAC,
        OutputType.CHIP_HISTONE,
        OutputType.CONTACT_MAPS,
    ],
    ontology_terms=None,
)
print("  ✅ PRL prediction complete")
gene_interval = genome.Interval(PRL_CHROM, PRL_START, PRL_END)

# ── Helper: filter tracks by ontology + mark name ────────────────────────────
def get_tracks(tdata, ontology, mark=None):
    mask = tdata.metadata['ontology_curie'] == ontology
    if mark:
        mask = mask & tdata.metadata['name'].str.contains(mark, na=False)
    m = mask.values
    return tdata.filter_tracks(m) if m.sum() > 0 else None

MSC  = 'CL:0000134'
UTER = 'UBERON:0000995'

Looking up PRL coordinates from Ensembl...
  PRL: chr6:22,287,244–22,302,826

Running AlphaGenome prediction for PRL...
  ✅ PRL prediction complete


In [7]:
# ── Figure A: PRL in mesenchymal stem cell with ±10% boundary extension visible
rna_msc   = get_tracks(prl_output.rna_seq,      MSC)
h27ac_msc = get_tracks(prl_output.chip_histone, MSC, 'H3K27ac')
h4me1_msc = get_tracks(prl_output.chip_histone, MSC, 'H3K4me1')
h4me3_msc = get_tracks(prl_output.chip_histone, MSC, 'H3K4me3')

# Extended display interval — shows gene + 10% upstream/downstream flanks
gene_len = PRL_END - PRL_START
ext = int(gene_len * 0.10)
display_interval = genome.Interval(PRL_CHROM, PRL_START - ext, PRL_END + ext)

components_A = []
if rna_msc:
    components_A.append(plot_components.Tracks(rna_msc,   ylabel_template='RNA-seq\n{biosample_name}'))
if h27ac_msc:
    components_A.append(plot_components.Tracks(h27ac_msc, ylabel_template='H3K27ac\n{biosample_name}'))
if h4me1_msc:
    components_A.append(plot_components.Tracks(h4me1_msc, ylabel_template='H3K4me1\n{biosample_name}'))
if h4me3_msc:
    components_A.append(plot_components.Tracks(h4me3_msc, ylabel_template='H3K4me3\n{biosample_name}'))

fig_A = plot_components.plot(
    components_A,
    interval=display_interval,
    title='AlphaGenome Predictions — PRL (Mesenchymal Stem Cell)\n±10% boundary extension shown',
)

# Grey box = gene body; dashed lines = gene boundaries
for ax in fig_A.axes:
    ax.axvspan(PRL_START, PRL_END, alpha=0.10, color='grey', zorder=0)
    ax.axvline(PRL_START, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.axvline(PRL_END,   color='black', linewidth=0.8, linestyle='--', alpha=0.5)

plt.savefig('outputs/figures/PRL_figureA_mesenchymal.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: outputs/figures/PRL_figureA_mesenchymal.png')
print(f'Gene body:      {PRL_CHROM}:{PRL_START:,}\u2013{PRL_END:,}  (grey shaded)')
print(f'Upstream flank: {ext:,} bp  |  Downstream flank: {ext:,} bp  (outside dashed lines)')


Saved: outputs/figures/PRL_figureA_mesenchymal.png
Gene body:      chr6:22,287,244–22,302,826  (grey shaded)
Upstream flank: 1,558 bp  |  Downstream flank: 1,558 bp  (outside dashed lines)


In [8]:
# ── Figure B: Two-condition layout with ±10% boundary extension visible
def build_condition_components(ontology, label):
    comps = []
    atac  = get_tracks(prl_output.atac,         ontology)
    h27ac = get_tracks(prl_output.chip_histone, ontology, 'H3K27ac')
    h4me1 = get_tracks(prl_output.chip_histone, ontology, 'H3K4me1')
    h4me3 = get_tracks(prl_output.chip_histone, ontology, 'H3K4me3')
    rna   = get_tracks(prl_output.rna_seq,      ontology)
    if atac:  comps.append(plot_components.Tracks(atac,  ylabel_template=f'ATAC-seq\n{label}'))
    if h27ac: comps.append(plot_components.Tracks(h27ac, ylabel_template=f'H3K27ac\n{label}'))
    if h4me1: comps.append(plot_components.Tracks(h4me1, ylabel_template=f'H3K4me1\n{label}'))
    if h4me3: comps.append(plot_components.Tracks(h4me3, ylabel_template=f'H3K4me3\n{label}'))
    if rna:   comps.append(plot_components.Tracks(rna,   ylabel_template=f'RNA-seq\n{label}'))
    return comps

gene_len = PRL_END - PRL_START
ext = int(gene_len * 0.10)
display_interval = genome.Interval(PRL_CHROM, PRL_START - ext, PRL_END + ext)

comps_cond1 = build_condition_components(MSC,  'Mesenchymal stem cell (Untreated proxy)')
comps_cond2 = build_condition_components(UTER, 'Uterus (Decidualized proxy)')

cm_meta = prl_output.contact_maps.metadata.reset_index(drop=True)
hela_mask = [r['biosample_name'] == 'HeLa-S3' for _, r in cm_meta.iterrows()]
cm_hela = prl_output.contact_maps.filter_tracks(hela_mask)

def add_gene_bounds(fig, color):
    for ax in fig.axes:
        ax.axvspan(PRL_START, PRL_END, alpha=0.10, color='grey', zorder=0)
        ax.axvline(PRL_START, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
        ax.axvline(PRL_END,   color='black', linewidth=0.8, linestyle='--', alpha=0.5)
    for ax in fig.axes:
        for coll in ax.collections:
            try: coll.set_facecolor(color); coll.set_edgecolor(color)
            except: pass
        for line in ax.lines:
            if line.get_linestyle() == '-': line.set_color(color)

fig1 = plot_components.plot(comps_cond1, interval=display_interval,
                             title='PRL — Mesenchymal Stem Cell (Untreated proxy)\n±10% flanks shown')
add_gene_bounds(fig1, '#2ecc71')
plt.savefig('outputs/figures/PRL_figureB_condition1_green.png', dpi=150, bbox_inches='tight')
plt.close()

fig2 = plot_components.plot(comps_cond2, interval=display_interval,
                             title='PRL — Uterus Tissue (Decidualized proxy)\n±10% flanks shown')
add_gene_bounds(fig2, '#8e44ad')
plt.savefig('outputs/figures/PRL_figureB_condition2_purple.png', dpi=150, bbox_inches='tight')
plt.close()

fig3 = plot_components.plot(
    [plot_components.ContactMaps(cm_hela, ylabel_template='{biosample_name}', cmap='Blues', vmax=1.0)],
    interval=genome.Interval(PRL_CHROM, p_start, p_end),
    title='PRL — Contact Map (HeLa-S3, proxy for pcHi-C)',
)
plt.savefig('outputs/figures/PRL_figureB_contactmap.png', dpi=150, bbox_inches='tight')
plt.close()

print('Saved all Figure B outputs')
print(f'Gene body shown with grey shading + dashed boundary lines')
print(f'Flanks: ±{ext:,} bp on each side')


Saved all Figure B outputs
Gene body shown with grey shading + dashed boundary lines
Flanks: ±1,558 bp on each side
